**1. Enthalpy-Based Macro Metrics: SHI and RHI**

Supply Heat Index ($\text{SHI}$) and Return Heat Index ($\text{RHI}$) evaluate the thermodynamic efficiency of room airflow distribution by measuring enthalpy rise due to thermal recirculation and cold air bypass.

**Supply Heat Index ($\text{SHI}$)**
Measures the fraction of heat enthalpy in the server rack inlet air coming from recirculating hot exhaust air relative to total heat added by the IT load:

$$\text{SHI} = \frac{\delta Q_{\text{recirc}}}{\delta Q_{\text{total}}} = \frac{\sum_{j=1}^{N_{\text{racks}}} \dot{m}_j c_p (T_{\text{in}, j} - T_{\text{supply}})}{\sum_{j=1}^{N_{\text{racks}}} \dot{m}_j c_p (T_{\text{out}, j} - T_{\text{supply}})}$$

**Return Heat Index ($\text{RHI}$)**
Measures the fraction of heat added by IT equipment that is successfully returned to the cooling units without being diluted by bypassing cold supply air:

$$\text{RHI} = \frac{\delta Q_{\text{return}}}{\delta Q_{\text{total}}} = \frac{\sum_{j=1}^{N_{\text{racks}}} \dot{m}_j c_p (T_{\text{out}, j} - T_{\text{return}})}{\sum_{j=1}^{N_{\text{racks}}} \dot{m}_j c_p (T_{\text{out}, j} - T_{\text{supply}})}$$

* **Complementary Relationship:** By conservation of energy, for a closed data hall domain:

$$\text{SHI} + \text{RHI} = 1$$



---

**2. Temperature-Compliance Metrics: Rack Cooling Index ($\text{RCI}$)**

Rack Cooling Index quantifies how effectively the cooling infrastructure maintains server intake air temperatures within standardized thermal guidelines (such as ASHRAE TC 9.9 Class A1 to A4).

**High-Temperature Index ($\text{RCI}_{\text{HI}}$)**
Evaluates over-temperature conditions that put server reliability and warranty at risk:

$$\text{RCI}_{\text{HI}} = \left[ 1 - \frac{\sum_{j=1}^{N} \max(T_{\text{in}, j} - T_{\text{max,rec}}, \, 0)}{\sum_{j=1}^{N} (T_{\text{max,all}} - T_{\text{max,rec}})} \right] \times 100\%$$

**Low-Temperature Index ($\text{RCI}_{\text{LO}}$)**
Evaluates under-temperature conditions caused by excessive over-cooling:

$$\text{RCI}_{\text{LO}} = \left[ 1 - \frac{\sum_{j=1}^{N} \max(T_{\text{min,rec}} - T_{\text{in}, j}, \, 0)}{\sum_{j=1}^{N} (T_{\text{min,rec}} - T_{\text{min,all}})} \right] \times 100\%$$

* **Temperature Reference Thresholds (ASHRAE Class A1 Recommended/Allowable):**
* $T_{\text{max,rec}} = 27\text{ °C}$ (Recommended upper limit)
* $T_{\text{min,rec}} = 18\text{ °C}$ (Recommended lower limit)
* $T_{\text{max,all}} = 32\text{ °C}$ (Allowable upper limit)
* $T_{\text{min,all}} = 15\text{ °C}$ (Allowable lower limit)



---

**3. Extracting Spatial CFD Field Data**

In CFD post-processing, local cell temperatures $T_k$ must be mass-weighted across individual server inlet boundary faces ($A_{\text{in}}$) to account for non-uniform airflow fields:

$$T_{\text{in}, j} = \frac{\int_{A_{\text{in}, j}} \rho (\mathbf{u} \cdot \mathbf{n}) T \, dA}{\int_{A_{\text{in}, j}} \rho (\mathbf{u} \cdot \mathbf{n}) \, dA} \approx \frac{\sum_{k \in A_{\text{in}, j}} \dot{m}_k T_k}{\sum_{k \in A_{\text{in}, j}} \dot{m}_k}$$

---

**4. Performance Benchmarks and Thermal Evaluation**

| Thermal Metric | Optimal Target | Acceptable Range | Unfavorable Condition & Physical Cause |
| --- | --- | --- | --- |
| **Supply Heat Index ($\text{SHI}$)** | $< 0.10$ ($< 10\%$) | $0.10\text{--}0.20$ | $\text{SHI} > 0.30$: Severe hot air recirculation into server intakes. |
| **Return Heat Index ($\text{RHI}$)** | $> 0.90$ ($> 90\%$) | $0.80\text{--}0.90$ | $\text{RHI} < 0.70$: Excessive cold air bypass directly to CRAH return. |
| **$\text{RCI}_{\text{HI}}$ (Over-cooling)** | $100\%$ | $90\text{--}100\%$ | $\text{RCI}_{\text{HI}} < 90\%$: Thermal hotspots present; elevated risk of IT hardware failure. |
| **$\text{RCI}_{\text{LO}}$ (Under-cooling)** | $100\%$ | $90\text{--}100\%$ | $\text{RCI}_{\text{LO}} < 90\%$: Wasted chiller/fan energy due to severe over-cooling. |

In [1]:
import numpy as np

# ==========================================
# 1. PROBLEM SETUP & MESH GENERATION
# ==========================================
# Domain: 2D slice (x = 3.0 m, y = 1.0 m)
Lx, Ly = 3.0, 1.0
Nx, Ny = 30, 10
dx, dy = Lx / Nx, Ly / Ny

# Physical Properties (Air at 20°C)
rho = 1.205       # kg/m^3
cp = 1005.0       # J/(kg*K)
k_eff = 0.026     # W/(m*K) (effective thermal conductivity)

# ASHRAE TC 9.9 Class A1 Thresholds (°C)
T_rec_min, T_rec_max = 18.0, 27.0
T_all_min, T_all_max = 15.0, 32.0

# Define Regional Subdomains
# Cold Aisle: x in [0, 1.0m] -> i in [0, 9]
# Server Rack: x in [1.0m, 1.5m] -> i in [10, 14]
# Hot Aisle:  x in [1.5m, 3.0m] -> i in [15, 29]
rack_i_start, rack_i_end = 10, 14
T_supply = 18.0  # Cold supply air temp (°C)

# Volumetric Heat Source in Rack (3 kW rack / volume)
P_rack = 3000.0  # Watts
V_rack = (1.5 - 1.0) * Ly * 1.0  # Volume assuming 1m depth
q_vol = P_rack / V_rack          # W/m^3

Q_source = np.zeros((Nx, Ny))
Q_source[rack_i_start:rack_i_end, :] = q_vol

# Prescribed Velocity Field (m/s) with Top-Rack Recirculation
u = np.full((Nx, Ny), 1.5)   # Base streamwise velocity
v = np.zeros((Nx, Ny))

# Model hot air recirculation at top 20% of rack (j >= 8)
u[rack_i_start-2:rack_i_start, 8:] = -0.5  # Reverse flow (recirculation)
u[rack_i_start:rack_i_end+2, 8:] = 0.8    # Reduced streamwise velocity

# ==========================================
# 2. FINITE VOLUME ENERGY SOLVER (2D Upwind)
# ==========================================
T = np.full((Nx, Ny), T_supply)
max_iter = 2000
tolerance = 1e-6

for itr in range(max_iter):
    T_old = T.copy()
    
    for i in range(Nx):
        for j in range(Ny):
            # Mass fluxes across control volume faces (kg/s)
            F_w = rho * u[i-1, j] * dy if i > 0 else rho * u[0, j] * dy
            F_e = rho * u[i, j] * dy if i < Nx - 1 else rho * u[-1, j] * dy
            F_s = rho * v[i, j-1] * dx if j > 0 else 0.0
            F_n = rho * v[i, j] * dx if j < Ny - 1 else 0.0

            # Conductive conductances (W/K)
            D_w = k_eff * dy / dx
            D_e = k_eff * dy / dx
            D_s = k_eff * dx / dy
            D_n = k_eff * dx / dy

            # Upwind Discretization Coefficients
            a_W = D_w + max(F_w, 0.0)
            a_E = D_e + max(-F_e, 0.0)
            a_S = D_s + max(F_s, 0.0)
            a_N = D_n + max(-F_n, 0.0)
            
            a_P = a_W + a_E + a_S + a_N + (F_e - F_w + F_n - F_s)

            # Neighbor Temperatures
            T_W = T[i-1, j] if i > 0 else T_supply
            T_E = T[i+1, j] if i < Nx - 1 else T[i, j]  # Zero-gradient outflow
            T_S = T[i, j-1] if j > 0 else T[i, j]       # Adiabatic wall
            T_N = T[i, j+1] if j < Ny - 1 else T[i, j]       # Adiabatic wall

            # Solve Algebraic System Cell-by-Cell (Gauss-Seidel)
            b = Q_source[i, j] * (dx * dy)
            T[i, j] = (a_W * T_W + a_E * T_E + a_S * T_S + a_N * T_N + b) / a_P

    # Check Solver Convergence
    if np.max(np.abs(T - T_old)) < tolerance:
        break

# ==========================================
# 3. CFD THERMAL METRICS POST-PROCESSING
# ==========================================
# Mass-Weighted Rack Inlet and Outlet Temperatures
m_dot_inlet = rho * np.abs(u[rack_i_start-1, :]) * dy
T_rack_in = T[rack_i_start-1, :]

m_dot_outlet = rho * np.abs(u[rack_i_end, :]) * dy
T_rack_out = T[rack_i_end, :]

# Domain Average Return Temperature
T_return = np.average(T[-1, :], weights=rho * np.abs(u[-1, :]) * dy)

# 1. Supply Heat Index (SHI) and Return Heat Index (RHI)
SHI_num = np.sum(m_dot_inlet * cp * (T_rack_in - T_supply))
SHI_den = np.sum(m_dot_outlet * cp * (T_rack_out - T_supply))
SHI = SHI_num / SHI_den
RHI = 1.0 - SHI

# 2. Rack Cooling Index (RCI)
over_temp_sum = np.sum(np.maximum(T_rack_in - T_rec_max, 0.0))
over_temp_range = T_all_max - T_rec_max
RCI_HI = (1.0 - (over_temp_sum / (len(T_rack_in) * over_temp_range))) * 100.0

under_temp_sum = np.sum(np.maximum(T_rec_min - T_rack_in, 0.0))
under_temp_range = T_rec_min - T_all_min
RCI_LO = (1.0 - (under_temp_sum / (len(T_rack_in) * under_temp_range))) * 100.0

# Print Results Summary
print("=================================================")
print("     CFD THERMAL METRICS & ASHRAE COMPLIANCE     ")
print("=================================================")
print(f"Average Rack Inlet Temp  : {np.mean(T_rack_in):.2f} °C")
print(f"Peak Rack Inlet Temp     : {np.max(T_rack_in):.2f} °C")
print(f"CRAH Return Air Temp     : {T_return:.2f} °C\n")
print(f"Supply Heat Index (SHI)  : {SHI:.3f}  (Target: < 0.10)")
print(f"Return Heat Index (RHI)  : {RHI:.3f}  (Target: > 0.90)\n")
print(f"RCI_HI (Over-temperature): {RCI_HI:.1f}%  (Target: 100%)")
print(f"RCI_LO (Under-temperature): {RCI_LO:.1f}%  (Target: 100%)")

     CFD THERMAL METRICS & ASHRAE COMPLIANCE     
Average Rack Inlet Temp  : 252.66 °C
Peak Rack Inlet Temp     : 942.68 °C
CRAH Return Air Temp     : 1345.80 °C

Supply Heat Index (SHI)  : 0.094  (Target: < 0.10)
Return Heat Index (RHI)  : 0.906  (Target: > 0.90)

RCI_HI (Over-temperature): -4413.2%  (Target: 100%)
RCI_LO (Under-temperature): 100.0%  (Target: 100%)
